# Volume Climax Pullback Survivors - Client Notebook

Ce notebook client regroupe en une seule vue les deux signaux pullback encore utiles apres les campagnes strictes :

- `M2K 1H` : survivor faible mais propre, verdict `weak_watchlist`
- `MGC 1H` : diversifiant opportuniste, mais instable en standalone, verdict `reject`

Le notebook recharge trois briques deja auditees :

1. le **survivor audit** pour les signaux stricts `M2K/MGC` ;
2. le **regime-gated audit** pour montrer pourquoi le gating MGC n'a pas ete retenu ;
3. l'**integration portefeuille** pour voir si le sleeve fixe `M2K + MGC` apporte quelque chose au book prop.

Objectif :

- garder un seul support client,
- montrer les verdicts stricts sans cherry-picking,
- rendre lisibles les deux signaux, le sleeve combine, et l'impact portefeuille.


In [1]:
import json
import math
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent

if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Impossible de retrouver la racine du repo depuis le notebook.")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import Markdown, display
from plotly.subplots import make_subplots

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 240)


def fmt_money(value):
    if value is None or pd.isna(value):
        return "n/a"
    return f"{float(value):,.1f} USD"


def fmt_pct(value, digits=1):
    if value is None or pd.isna(value):
        return "n/a"
    return f"{float(value):.{digits}f}%"


def fmt_float(value, digits=3):
    if value is None or pd.isna(value):
        return "n/a"
    return f"{float(value):.{digits}f}"


def safe_corr(left, right):
    left = pd.to_numeric(left, errors="coerce")
    right = pd.to_numeric(right, errors="coerce")
    if left.count() < 2 or right.count() < 2:
        return float("nan")
    return float(left.corr(right))


def build_equity(frame, pnl_col):
    out = frame.copy()
    out[pnl_col] = pd.to_numeric(out[pnl_col], errors="coerce").fillna(0.0)
    out["equity"] = out[pnl_col].cumsum()
    out["peak"] = out["equity"].cummax()
    out["drawdown"] = out["equity"] - out["peak"]
    return out


In [2]:
SURVIVOR_EXPORT_ROOT = ROOT / r"export\volume_climax_pullback_survivor_audit_20260521_091448"
REGIME_EXPORT_ROOT = ROOT / r"export\volume_climax_pullback_regime_gated_portfolio_20260521_233956"
INTEGRATION_EXPORT_ROOT = ROOT / r"export\volume_climax_pullback_portfolio_integration_20260522_003708"

PRIMARY_SYMBOLS = ["M2K", "MGC"]
NEGATIVE_CONTROL = "MNQ"
SIGNAL_TIMEFRAME = "1H"

STRICT_SIGNAL_LABELS = {
    "M2K": "M2K 1H survivor",
    "MGC": "MGC 1H opportunistic diversifier",
    "MNQ": "MNQ 1H negative control",
}

STRICT_SLEEVE_NAME = "m2k_mgc_equal_weight"
STRICT_SLEEVE_FALLBACK = "m2k_mgc_capped_equal"
M2K_ONLY_PORTFOLIO = "m2k_only"
MGC_ONLY_PORTFOLIO = "mgc_only"
BEST_REGIME_PORTFOLIO = "strict_best_regime_gated"

BASELINE_PORTFOLIO = "baseline_only"
PULLBACK_PORTFOLIO = "pullback_m2k_mgc_only"
INTEGRATED_PORTFOLIO = "baseline_plus_pullback_equal_notional"
INTEGRATED_M2K_ONLY = "baseline_plus_m2k_only_equal_notional"

PLOT_TEMPLATE = "plotly_white"

required_paths = {
    "survivor_summary": SURVIVOR_EXPORT_ROOT / "strict_wfa_summary.csv",
    "survivor_fold_breakdown": SURVIVOR_EXPORT_ROOT / "strict_wfa_fold_breakdown.csv",
    "survivor_portfolio_summary": SURVIVOR_EXPORT_ROOT / "strict_portfolio_summary.csv",
    "survivor_portfolio_daily": SURVIVOR_EXPORT_ROOT / "strict_portfolio_daily_returns.csv",
    "survivor_selection": SURVIVOR_EXPORT_ROOT / "config_selection_by_fold.csv",
    "survivor_cluster_stability": SURVIVOR_EXPORT_ROOT / "cluster_stability_summary.csv",
    "survivor_local_stability": SURVIVOR_EXPORT_ROOT / "local_parameter_stability.csv",
    "survivor_monthly": SURVIVOR_EXPORT_ROOT / "monthly_pnl.csv",
    "survivor_yearly": SURVIVOR_EXPORT_ROOT / "yearly_pnl.csv",
    "survivor_trade_concentration": SURVIVOR_EXPORT_ROOT / "trade_concentration.csv",
    "regime_summary": REGIME_EXPORT_ROOT / "strict_regime_wfa_summary.csv",
    "regime_rules": REGIME_EXPORT_ROOT / "selected_regime_rule_by_fold.csv",
    "regime_retention": REGIME_EXPORT_ROOT / "mgc_regime_retention_summary.csv",
    "integration_summary": INTEGRATION_EXPORT_ROOT / "portfolio_summary.csv",
    "integration_daily": INTEGRATION_EXPORT_ROOT / "daily_pnl_aligned.csv",
    "integration_corr": INTEGRATION_EXPORT_ROOT / "portfolio_correlation.csv",
    "integration_incremental": INTEGRATION_EXPORT_ROOT / "incremental_metrics.csv",
    "integration_bootstrap": INTEGRATION_EXPORT_ROOT / "bootstrap_summary.csv",
    "integration_prop": INTEGRATION_EXPORT_ROOT / "prop_constraint_summary.csv",
}

missing = [name for name, path in required_paths.items() if not path.exists()]
if missing:
    raise FileNotFoundError(f"Fichiers manquants pour le notebook: {missing}")

display(Markdown("### Parametrage client"))
display(pd.DataFrame(
    [
        {"parameter": "SURVIVOR_EXPORT_ROOT", "value": str(SURVIVOR_EXPORT_ROOT)},
        {"parameter": "REGIME_EXPORT_ROOT", "value": str(REGIME_EXPORT_ROOT)},
        {"parameter": "INTEGRATION_EXPORT_ROOT", "value": str(INTEGRATION_EXPORT_ROOT)},
        {"parameter": "PRIMARY_SYMBOLS", "value": ", ".join(PRIMARY_SYMBOLS)},
        {"parameter": "NEGATIVE_CONTROL", "value": NEGATIVE_CONTROL},
        {"parameter": "SIGNAL_TIMEFRAME", "value": SIGNAL_TIMEFRAME},
        {"parameter": "STRICT_SLEEVE_NAME", "value": STRICT_SLEEVE_NAME},
        {"parameter": "BASELINE_PORTFOLIO", "value": BASELINE_PORTFOLIO},
        {"parameter": "INTEGRATED_PORTFOLIO", "value": INTEGRATED_PORTFOLIO},
        {"parameter": "PLOT_TEMPLATE", "value": PLOT_TEMPLATE},
    ]
))


### Parametrage client

,parameter,value
0,SURVIVOR_EXPORT_ROOT,D:\Business\Trading\VSCODE\algo-trading-intrad...
1,REGIME_EXPORT_ROOT,D:\Business\Trading\VSCODE\algo-trading-intrad...
2,INTEGRATION_EXPORT_ROOT,D:\Business\Trading\VSCODE\algo-trading-intrad...
3,PRIMARY_SYMBOLS,"M2K, MGC"
4,NEGATIVE_CONTROL,MNQ
5,SIGNAL_TIMEFRAME,1H
6,STRICT_SLEEVE_NAME,m2k_mgc_equal_weight
7,BASELINE_PORTFOLIO,baseline_only
8,INTEGRATED_PORTFOLIO,baseline_plus_pullback_equal_notional
9,PLOT_TEMPLATE,plotly_white


In [3]:
strict_wfa_summary = pd.read_csv(SURVIVOR_EXPORT_ROOT / "strict_wfa_summary.csv")
strict_wfa_fold_breakdown = pd.read_csv(SURVIVOR_EXPORT_ROOT / "strict_wfa_fold_breakdown.csv")
strict_portfolio_summary = pd.read_csv(SURVIVOR_EXPORT_ROOT / "strict_portfolio_summary.csv")
strict_portfolio_daily = pd.read_csv(
    SURVIVOR_EXPORT_ROOT / "strict_portfolio_daily_returns.csv",
    parse_dates=["session_date"],
)
config_selection_by_fold = pd.read_csv(SURVIVOR_EXPORT_ROOT / "config_selection_by_fold.csv")
cluster_stability_summary = pd.read_csv(SURVIVOR_EXPORT_ROOT / "cluster_stability_summary.csv")
local_parameter_stability = pd.read_csv(SURVIVOR_EXPORT_ROOT / "local_parameter_stability.csv")
survivor_monthly_pnl = pd.read_csv(SURVIVOR_EXPORT_ROOT / "monthly_pnl.csv")
survivor_yearly_pnl = pd.read_csv(SURVIVOR_EXPORT_ROOT / "yearly_pnl.csv")
survivor_trade_concentration = pd.read_csv(SURVIVOR_EXPORT_ROOT / "trade_concentration.csv")

strict_regime_wfa_summary = pd.read_csv(REGIME_EXPORT_ROOT / "strict_regime_wfa_summary.csv")
selected_regime_rule_by_fold = pd.read_csv(REGIME_EXPORT_ROOT / "selected_regime_rule_by_fold.csv")
mgc_regime_retention_summary = pd.read_csv(REGIME_EXPORT_ROOT / "mgc_regime_retention_summary.csv")

portfolio_summary = pd.read_csv(INTEGRATION_EXPORT_ROOT / "portfolio_summary.csv")
daily_pnl_aligned = pd.read_csv(INTEGRATION_EXPORT_ROOT / "daily_pnl_aligned.csv", parse_dates=["session_date"])
portfolio_correlation = pd.read_csv(INTEGRATION_EXPORT_ROOT / "portfolio_correlation.csv")
incremental_metrics = pd.read_csv(INTEGRATION_EXPORT_ROOT / "incremental_metrics.csv")
bootstrap_summary = pd.read_csv(INTEGRATION_EXPORT_ROOT / "bootstrap_summary.csv")
prop_constraint_summary = pd.read_csv(INTEGRATION_EXPORT_ROOT / "prop_constraint_summary.csv")

strict_signal_view = strict_wfa_summary.loc[
    (strict_wfa_summary["signal_timeframe"].astype(str) == SIGNAL_TIMEFRAME)
    & (strict_wfa_summary["symbol"].astype(str).isin(PRIMARY_SYMBOLS + [NEGATIVE_CONTROL]))
].copy()
strict_signal_view["display_name"] = strict_signal_view["symbol"].astype(str).map(STRICT_SIGNAL_LABELS)

strict_fold_view = strict_wfa_fold_breakdown.loc[
    (strict_wfa_fold_breakdown["signal_timeframe"].astype(str) == SIGNAL_TIMEFRAME)
    & (strict_wfa_fold_breakdown["symbol"].astype(str).isin(PRIMARY_SYMBOLS + [NEGATIVE_CONTROL]))
].copy()
strict_fold_view["display_name"] = strict_fold_view["symbol"].astype(str).map(STRICT_SIGNAL_LABELS)

selection_view = config_selection_by_fold.loc[
    config_selection_by_fold["symbol"].astype(str).isin(PRIMARY_SYMBOLS + [NEGATIVE_CONTROL])
].copy()

cluster_view = cluster_stability_summary.loc[
    cluster_stability_summary["symbol"].astype(str).isin(PRIMARY_SYMBOLS + [NEGATIVE_CONTROL])
].copy()

local_view = local_parameter_stability.loc[
    local_parameter_stability["symbol"].astype(str).isin(PRIMARY_SYMBOLS + [NEGATIVE_CONTROL])
].copy()

strict_portfolios = strict_portfolio_summary.copy()
strict_portfolio_daily["session_date"] = pd.to_datetime(strict_portfolio_daily["session_date"], errors="coerce").dt.normalize()
daily_pnl_aligned["session_date"] = pd.to_datetime(daily_pnl_aligned["session_date"], errors="coerce").dt.normalize()

regime_entity_col = "entity_id" if "entity_id" in strict_regime_wfa_summary.columns else "entity_name"

strict_sleeve_row = strict_portfolios.loc[strict_portfolios["portfolio_name"].astype(str) == STRICT_SLEEVE_NAME]
if strict_sleeve_row.empty:
    strict_sleeve_row = strict_portfolios.loc[strict_portfolios["portfolio_name"].astype(str) == STRICT_SLEEVE_FALLBACK]
if strict_sleeve_row.empty:
    raise ValueError("Impossible de retrouver le sleeve strict M2K+MGC dans strict_portfolio_summary.csv")
strict_sleeve_row = strict_sleeve_row.iloc[0]

m2k_row = strict_portfolios.loc[strict_portfolios["portfolio_name"].astype(str) == M2K_ONLY_PORTFOLIO].iloc[0]
mgc_row = strict_portfolios.loc[strict_portfolios["portfolio_name"].astype(str) == MGC_ONLY_PORTFOLIO].iloc[0]
best_regime_row = strict_regime_wfa_summary.loc[
    strict_regime_wfa_summary[regime_entity_col].astype(str) == BEST_REGIME_PORTFOLIO
].iloc[0]

baseline_row = portfolio_summary.loc[
    (portfolio_summary["portfolio_name"].astype(str) == BASELINE_PORTFOLIO)
    & (portfolio_summary["scope"].astype(str) == "defined_oos")
].iloc[0]
pullback_row = portfolio_summary.loc[
    (portfolio_summary["portfolio_name"].astype(str) == PULLBACK_PORTFOLIO)
    & (portfolio_summary["scope"].astype(str) == "defined_oos")
].iloc[0]
integrated_row = portfolio_summary.loc[
    (portfolio_summary["portfolio_name"].astype(str) == INTEGRATED_PORTFOLIO)
    & (portfolio_summary["scope"].astype(str) == "defined_oos")
].iloc[0]
integrated_m2k_row = portfolio_summary.loc[
    (portfolio_summary["portfolio_name"].astype(str) == INTEGRATED_M2K_ONLY)
    & (portfolio_summary["scope"].astype(str) == "defined_oos")
].iloc[0]

incremental_equal = incremental_metrics.loc[incremental_metrics["portfolio_name"].astype(str) == INTEGRATED_PORTFOLIO].iloc[0]
bootstrap_equal = bootstrap_summary.loc[bootstrap_summary["portfolio_name"].astype(str) == INTEGRATED_PORTFOLIO].iloc[0]
bootstrap_baseline = bootstrap_summary.loc[bootstrap_summary["portfolio_name"].astype(str) == BASELINE_PORTFOLIO].iloc[0]
bootstrap_pullback = bootstrap_summary.loc[bootstrap_summary["portfolio_name"].astype(str) == PULLBACK_PORTFOLIO].iloc[0]
prop_equal = prop_constraint_summary.loc[prop_constraint_summary["portfolio_name"].astype(str) == INTEGRATED_PORTFOLIO].iloc[0]
prop_baseline = prop_constraint_summary.loc[prop_constraint_summary["portfolio_name"].astype(str) == BASELINE_PORTFOLIO].iloc[0]

correlation_equal = portfolio_correlation.loc[
    (portfolio_correlation["left_portfolio"].astype(str) == BASELINE_PORTFOLIO)
    & (portfolio_correlation["right_portfolio"].astype(str) == PULLBACK_PORTFOLIO)
].iloc[0]

strict_sleeve_daily = strict_portfolio_daily.loc[
    strict_portfolio_daily["portfolio_name"].astype(str).isin([STRICT_SLEEVE_NAME, STRICT_SLEEVE_FALLBACK])
].copy()
strict_sleeve_daily = strict_sleeve_daily.sort_values("session_date").drop_duplicates("session_date", keep="last")

integration_curve = daily_pnl_aligned.loc[daily_pnl_aligned["oos_mask"].fillna(False)].copy()
integration_curve["baseline_equity"] = pd.to_numeric(integration_curve["baseline_daily_pnl_usd"], errors="coerce").fillna(0.0).cumsum()
integration_curve["pullback_equity"] = pd.to_numeric(integration_curve["pullback_daily_pnl_usd"], errors="coerce").fillna(0.0).cumsum()
integration_curve["integrated_equity"] = (
    pd.to_numeric(integration_curve["baseline_daily_pnl_usd"], errors="coerce").fillna(0.0)
    + pd.to_numeric(integration_curve["pullback_daily_pnl_usd"], errors="coerce").fillna(0.0)
).cumsum()

display(Markdown(f"**Survivor export**: `{SURVIVOR_EXPORT_ROOT}`"))
display(Markdown(f"**Regime export**: `{REGIME_EXPORT_ROOT}`"))
display(Markdown(f"**Integration export**: `{INTEGRATION_EXPORT_ROOT}`"))


**Survivor export**: `D:\Business\Trading\VSCODE\algo-trading-intraday-research\export\volume_climax_pullback_survivor_audit_20260521_091448`

**Regime export**: `D:\Business\Trading\VSCODE\algo-trading-intraday-research\export\volume_climax_pullback_regime_gated_portfolio_20260521_233956`

**Integration export**: `D:\Business\Trading\VSCODE\algo-trading-intraday-research\export\volume_climax_pullback_portfolio_integration_20260522_003708`

In [4]:
display(Markdown("## 1. Executive Summary"))

summary_lines = [
    f"- `M2K 1H` reste le seul survivor standalone non rejete : net strict WFA `{fmt_money(strict_signal_view.loc[strict_signal_view['symbol'] == 'M2K', 'total_test_net_pnl'].iloc[0])}` | PF `{fmt_float(strict_signal_view.loc[strict_signal_view['symbol'] == 'M2K', 'test_profit_factor'].iloc[0])}` | folds positifs `{int(strict_signal_view.loc[strict_signal_view['symbol'] == 'M2K', 'positive_folds'].iloc[0])}/{int(strict_signal_view.loc[strict_signal_view['symbol'] == 'M2K', 'fold_count'].iloc[0])}` | verdict `{strict_signal_view.loc[strict_signal_view['symbol'] == 'M2K', 'verdict'].iloc[0]}`.",
    f"- `MGC 1H` garde un net eleve mais trop concentre regime/folds : net `{fmt_money(strict_signal_view.loc[strict_signal_view['symbol'] == 'MGC', 'total_test_net_pnl'].iloc[0])}` | PF `{fmt_float(strict_signal_view.loc[strict_signal_view['symbol'] == 'MGC', 'test_profit_factor'].iloc[0])}` | folds positifs `{int(strict_signal_view.loc[strict_signal_view['symbol'] == 'MGC', 'positive_folds'].iloc[0])}/{int(strict_signal_view.loc[strict_signal_view['symbol'] == 'MGC', 'fold_count'].iloc[0])}` | verdict `{strict_signal_view.loc[strict_signal_view['symbol'] == 'MGC', 'verdict'].iloc[0]}`.",
    f"- Sleeve strict `M2K + MGC` equal-weight : net `{fmt_money(strict_sleeve_row['net_pnl'])}` | PF `{fmt_float(strict_sleeve_row['profit_factor'])}` | maxDD `{fmt_money(strict_sleeve_row['max_drawdown'])}` | verdict `{strict_sleeve_row['verdict']}`.",
    f"- Regime gating MGC non retenu : meilleur portefeuille gate net `{fmt_money(best_regime_row['net_pnl'])}` versus sleeve raw `{fmt_money(strict_sleeve_row['net_pnl'])}`.",
    f"- Integration au book baseline : baseline `{fmt_money(baseline_row['net_pnl'])}` -> baseline + sleeve `{fmt_money(integrated_row['net_pnl'])}` | incremental `{fmt_money(incremental_equal['incremental_net_pnl_vs_baseline'])}` | corr pullback vs baseline `{fmt_float(correlation_equal['correlation'])}`.",
]
display(Markdown("\n".join(summary_lines)))

executive_table = strict_signal_view[
    [
        "display_name",
        "total_test_net_pnl",
        "test_profit_factor",
        "positive_folds",
        "fold_count",
        "avg_trade",
        "max_drawdown",
        "monthly_positive_ratio",
        "verdict",
    ]
].copy()
display(executive_table.round(3))


## 1. Executive Summary

- `M2K 1H` reste le seul survivor standalone non rejete : net strict WFA `474.8 USD` | PF `1.913` | folds positifs `4/5` | verdict `weak_watchlist`.
- `MGC 1H` garde un net eleve mais trop concentre regime/folds : net `1,217.2 USD` | PF `3.489` | folds positifs `2/5` | verdict `reject`.
- Sleeve strict `M2K + MGC` equal-weight : net `846.0 USD` | PF `2.677` | maxDD `-107.8 USD` | verdict `watchlist`.
- Regime gating MGC non retenu : meilleur portefeuille gate net `800.3 USD` versus sleeve raw `846.0 USD`.
- Integration au book baseline : baseline `24,714.0 USD` -> baseline + sleeve `24,927.3 USD` | incremental `213.3 USD` | corr pullback vs baseline `-0.056`.

,display_name,total_test_net_pnl,test_profit_factor,positive_folds,fold_count,avg_trade,max_drawdown,monthly_positive_ratio,verdict
0,M2K 1H survivor,474.775,1.913,4,5,12.174,-133.125,0.440,weak_watchlist
1,MGC 1H opportunistic diversifier,1217.200,3.489,2,5,52.922,-265.000,0.625,reject
2,MNQ 1H negative control,-878.125,0.865,2,5,-7.570,-2260.775,0.367,reject


In [5]:
display(Markdown("## 2. The Two Signals"))

signal_card = strict_signal_view[
    [
        "symbol",
        "display_name",
        "total_test_trades",
        "total_test_net_pnl",
        "test_profit_factor",
        "avg_trade",
        "max_drawdown",
        "positive_folds",
        "fold_count",
        "train_score_test_corr",
        "selected_family_counts",
        "selected_cluster_counts",
        "verdict",
    ]
].copy()
display(signal_card.round(3))

selected_columns = [
    "symbol",
    "fold_id",
    "config_id",
    "family",
    "cluster_id",
    "filter_name",
    "stop_multiplier",
    "target_multiplier",
    "entry_delay_minutes",
    "train_robust_score",
    "train_net_pnl",
    "train_profit_factor",
    "train_trades",
    "selected_in_fold",
]
signal_selection = selection_view.loc[
    selection_view["symbol"].astype(str).isin(PRIMARY_SYMBOLS),
    selected_columns,
].copy()
display(Markdown("### Fold selections actually used"))
display(signal_selection.round(3))


## 2. The Two Signals

,symbol,display_name,total_test_trades,total_test_net_pnl,test_profit_factor,avg_trade,max_drawdown,positive_folds,fold_count,train_score_test_corr,selected_family_counts,selected_cluster_counts,verdict
0,M2K,M2K 1H survivor,39,474.775,1.913,12.174,-133.125,4,5,0.684,"{""delay_adverse_filter"": 4, ""delay_stop_zone_f...","{""m2k_1h_adverse_core"": 4, ""m2k_1h_stop_zone_d...",weak_watchlist
1,MGC,MGC 1H opportunistic diversifier,23,1217.200,3.489,52.922,-265.000,2,5,0.700,"{""delay_stop_zone_filter"": 2, ""none_baseline"":...","{""mgc_1h_none_core"": 2, ""mgc_1h_raw_baseline"":...",reject
2,MNQ,MNQ 1H negative control,116,-878.125,0.865,-7.570,-2260.775,2,5,-0.027,"{""delay_stop_zone_filter"": 1, ""none_baseline"": 4}","{""mnq_1h_negative_control_none"": 4, ""mnq_1h_ne...",reject


### Fold selections actually used

,symbol,fold_id,config_id,family,cluster_id,filter_name,stop_multiplier,target_multiplier,entry_delay_minutes,train_robust_score,train_net_pnl,train_profit_factor,train_trades,selected_in_fold
0,M2K,fold_1,m2k_1h_delay_stop_zone_filter_require_no_stop_...,delay_stop_zone_filter,m2k_1h_stop_zone_diag,require_no_stop_zone_touch_before_entry,0.75,2.0,15,0.275,32.000,1.209,10,True
1,M2K,fold_1,m2k_1h_delay_adverse_filter_avoid_immediate_ad...,delay_adverse_filter,m2k_1h_adverse_core,avoid_immediate_adverse_move,0.75,2.5,5,0.186,12.250,1.024,23,False
2,M2K,fold_1,m2k_1h_delay_adverse_filter_avoid_immediate_ad...,delay_adverse_filter,m2k_1h_adverse_core,avoid_immediate_adverse_move,1.00,2.5,5,0.183,10.375,1.018,23,False
3,M2K,fold_1,m2k_1h_delay_adverse_filter_avoid_immediate_ad...,delay_adverse_filter,m2k_1h_adverse_core,avoid_immediate_adverse_move,0.75,2.0,15,0.182,16.375,1.043,19,False
4,M2K,fold_1,m2k_1h_delay_adverse_filter_avoid_immediate_ad...,delay_adverse_filter,m2k_1h_adverse_core,avoid_immediate_adverse_move,0.75,2.5,15,0.182,16.375,1.043,19,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1750,MGC,fold_5,mgc_1h_delay_adverse_filter_avoid_immediate_ad...,delay_adverse_filter,mgc_1h_adverse_diag,avoid_immediate_adverse_move,1.25,3.0,30,-0.342,-212.250,0.197,11,False
1751,MGC,fold_5,mgc_1h_delay_adverse_filter_avoid_immediate_ad...,delay_adverse_filter,mgc_1h_adverse_diag,avoid_immediate_adverse_move,1.25,2.5,30,-0.356,-359.250,0.177,14,False
1752,MGC,fold_5,mgc_1h_delay_adverse_filter_avoid_immediate_ad...,delay_adverse_filter,mgc_1h_adverse_diag,avoid_immediate_adverse_move,1.25,3.0,30,-0.356,-359.250,0.177,14,False
1753,MGC,fold_5,mgc_1h_delay_adverse_filter_avoid_immediate_ad...,delay_adverse_filter,mgc_1h_adverse_diag,avoid_immediate_adverse_move,1.00,2.5,30,-0.358,-366.000,0.175,14,False


In [6]:
display(Markdown("## 3. Fold-by-Fold and Sleeve Curves"))

fold_fig = px.bar(
    strict_fold_view.loc[strict_fold_view["symbol"].astype(str).isin(PRIMARY_SYMBOLS)],
    x="fold_id",
    y="test_net_pnl",
    color="display_name",
    barmode="group",
    title="Strict WFA test PnL by fold",
)
fold_fig.update_layout(template=PLOT_TEMPLATE, width=1150, height=450)
fold_fig.show()

curve_fig = make_subplots(
    rows=2,
    cols=1,
    vertical_spacing=0.12,
    subplot_titles=("Strict sleeve stitched equity", "Portfolio integration OOS equity"),
)

for portfolio_name, label, color in [
    (M2K_ONLY_PORTFOLIO, "M2K only", "#2563eb"),
    (MGC_ONLY_PORTFOLIO, "MGC only", "#a855f7"),
    (STRICT_SLEEVE_NAME, "M2K + MGC strict sleeve", "#f59e0b"),
    (STRICT_SLEEVE_FALLBACK, "M2K + MGC strict sleeve", "#f59e0b"),
]:
    frame = strict_portfolio_daily.loc[strict_portfolio_daily["portfolio_name"].astype(str) == portfolio_name].copy()
    if frame.empty:
        continue
    frame = frame.sort_values("session_date")
    curve_fig.add_trace(
        go.Scatter(
            x=frame["session_date"],
            y=frame["equity"],
            mode="lines",
            name=label,
            line=dict(color=color, width=2.5 if "strict sleeve" in label else 1.9),
        ),
        row=1,
        col=1,
    )

for name, y_col, color in [
    ("Baseline only", "baseline_equity", "#475569"),
    ("Pullback sleeve only", "pullback_equity", "#f59e0b"),
    ("Baseline + pullback", "integrated_equity", "#16a34a"),
]:
    curve_fig.add_trace(
        go.Scatter(
            x=integration_curve["session_date"],
            y=integration_curve[y_col],
            mode="lines",
            name=name,
            line=dict(color=color, width=2.6 if name == "Baseline + pullback" else 2.0),
        ),
        row=2,
        col=1,
    )

curve_fig.update_yaxes(title_text="Equity (USD)", row=1, col=1)
curve_fig.update_yaxes(title_text="OOS cumulative pnl (USD)", row=2, col=1)
curve_fig.update_xaxes(title_text="Session date", row=2, col=1)
curve_fig.update_layout(template=PLOT_TEMPLATE, height=950, width=1450, legend=dict(orientation="h", y=-0.10, x=0.0))
curve_fig.show()


## 3. Fold-by-Fold and Sleeve Curves

In [7]:
display(Markdown("## 4. Stability Around Parameters"))

cluster_table = cluster_view[
    [
        "symbol",
        "cluster_id",
        "family",
        "configs",
        "median_is_net_pnl",
        "median_oos_net_pnl",
        "median_fixed_wfa_net_pnl",
        "pct_configs_positive_oos",
        "pct_configs_positive_fixed_wfa",
        "selected_in_any_fold",
    ]
].copy()
display(cluster_table.round(3))

local_focus = local_view.loc[
    local_view["symbol"].astype(str).isin(PRIMARY_SYMBOLS)
    & local_view["rank_is"].fillna(999999).astype(int).le(15)
].copy()
display(Markdown("### Local neighborhood around the best ranks"))
display(
    local_focus[
        [
            "symbol",
            "config_id",
            "family",
            "cluster_id",
            "stop_multiplier",
            "target_multiplier",
            "entry_delay_minutes",
            "variant_time_stop_bars",
            "stop_zone_fraction",
            "adverse_window_minutes",
            "max_adverse_ticks",
            "robust_score_is",
            "net_pnl_oos",
            "fixed_wfa_net_pnl",
            "neighbor_median_oos_pnl",
            "neighbor_positive_fold_ratio",
        ]
    ].round(3)
)

scatter = px.scatter(
    local_focus,
    x="robust_score_is",
    y="fixed_wfa_net_pnl",
    color="symbol",
    hover_name="config_id",
    symbol="family",
    title="Train robustness vs fixed WFA net pnl",
)
scatter.update_layout(template=PLOT_TEMPLATE, width=1200, height=550)
scatter.show()


## 4. Stability Around Parameters

,symbol,cluster_id,family,configs,median_is_net_pnl,median_oos_net_pnl,median_fixed_wfa_net_pnl,pct_configs_positive_oos,pct_configs_positive_fixed_wfa,selected_in_any_fold
0,M2K,m2k_1h_adverse_core,delay_adverse_filter,64,-80.369,262.950,388.000,1.000,1.000,3
1,M2K,m2k_1h_none_local,none_baseline,36,-1002.794,743.719,410.013,1.000,0.944,0
2,M2K,m2k_1h_raw_baseline,raw_hybrid,2,-1336.187,286.475,97.513,1.000,1.000,0
3,M2K,m2k_1h_stop_zone_diag,delay_stop_zone_filter,64,-774.975,557.813,221.531,0.891,0.781,1
4,MGC,mgc_1h_adverse_diag,delay_adverse_filter,64,-1.125,-61.750,-32.375,0.094,0.406,0
5,MGC,mgc_1h_none_core,none_baseline,54,215.375,797.500,733.500,0.889,0.889,1
6,MGC,mgc_1h_raw_baseline,raw_hybrid,3,-316.300,753.550,862.600,1.000,1.000,1
7,MGC,mgc_1h_stop_zone_core,delay_stop_zone_filter,64,-158.650,329.625,409.875,0.875,0.875,1
8,MNQ,mnq_1h_negative_control_adverse,delay_adverse_filter,32,73.300,503.575,108.500,1.000,0.812,0
9,MNQ,mnq_1h_negative_control_none,none_baseline,8,1287.163,380.431,606.612,0.750,1.000,2


### Local neighborhood around the best ranks

,symbol,config_id,family,cluster_id,stop_multiplier,target_multiplier,entry_delay_minutes,variant_time_stop_bars,stop_zone_fraction,adverse_window_minutes,max_adverse_ticks,robust_score_is,net_pnl_oos,fixed_wfa_net_pnl,neighbor_median_oos_pnl,neighbor_positive_fold_ratio
0,M2K,m2k_1h_delay_adverse_filter_avoid_immediate_ad...,delay_adverse_filter,m2k_1h_adverse_core,0.75,2.0,15,3,NaN,5.0,8.0,0.789,354.400,814.275,262.95,0.709
1,M2K,m2k_1h_delay_adverse_filter_avoid_immediate_ad...,delay_adverse_filter,m2k_1h_adverse_core,0.75,2.0,15,3,NaN,5.0,12.0,0.773,351.275,679.275,262.95,0.709
2,M2K,m2k_1h_delay_adverse_filter_avoid_immediate_ad...,delay_adverse_filter,m2k_1h_adverse_core,0.75,2.0,15,3,NaN,10.0,12.0,0.744,313.150,560.775,262.95,0.709
3,M2K,m2k_1h_delay_adverse_filter_avoid_immediate_ad...,delay_adverse_filter,m2k_1h_adverse_core,1.00,2.0,15,3,NaN,5.0,8.0,0.725,300.900,751.650,262.95,0.709
4,M2K,m2k_1h_delay_adverse_filter_avoid_immediate_ad...,delay_adverse_filter,m2k_1h_adverse_core,0.75,3.0,15,3,NaN,5.0,12.0,0.554,314.875,498.700,262.95,0.709
5,M2K,m2k_1h_delay_adverse_filter_avoid_immediate_ad...,delay_adverse_filter,m2k_1h_adverse_core,0.75,2.5,15,3,NaN,5.0,12.0,0.545,311.438,475.438,262.95,0.709
6,M2K,m2k_1h_delay_adverse_filter_avoid_immediate_ad...,delay_adverse_filter,m2k_1h_adverse_core,0.75,3.0,15,3,NaN,5.0,8.0,0.540,318.000,633.700,262.95,0.709
7,M2K,m2k_1h_delay_adverse_filter_avoid_immediate_ad...,delay_adverse_filter,m2k_1h_adverse_core,0.75,2.5,15,3,NaN,5.0,8.0,0.527,314.563,610.438,262.95,0.709
8,M2K,m2k_1h_delay_adverse_filter_avoid_immediate_ad...,delay_adverse_filter,m2k_1h_adverse_core,0.75,3.0,15,3,NaN,10.0,12.0,0.527,276.750,380.200,262.95,0.709
9,M2K,m2k_1h_delay_adverse_filter_avoid_immediate_ad...,delay_adverse_filter,m2k_1h_adverse_core,1.00,2.0,15,3,NaN,5.0,12.0,0.519,261.400,508.150,262.95,0.709


In [8]:
display(Markdown("## 5. Why Regime Gating Was Not Retained"))

regime_compare = strict_regime_wfa_summary.copy()
regime_compare["entity_label"] = regime_compare[regime_entity_col].astype(str)
display(regime_compare.round(3))

display(Markdown("### Selected regime rule by fold"))
display(
    selected_regime_rule_by_fold[
        [
            "fold_id",
            "rule_id",
            "family",
            "allocation_scheme",
            "train_score",
            "train_net_pnl",
            "train_profit_factor",
            "train_trades",
            "mgc_retention_rate_train",
        ]
    ].round(3)
)

retention_fig = px.bar(
    mgc_regime_retention_summary,
    x="fold_id",
    y=["mgc_test_retention_rate", "mgc_train_retention_rate"],
    barmode="group",
    title="MGC retention under selected regime rules",
)
retention_fig.update_layout(template=PLOT_TEMPLATE, width=1100, height=450)
retention_fig.show()


## 5. Why Regime Gating Was Not Retained

,entity_name,selection_basis,deployable,net_pnl,profit_factor,trades,positive_folds,max_drawdown,max_daily_drawdown,win_rate,avg_trade,median_trade,monthly_hit_rate,active_months,mgc_trade_retention_rate,mgc_contribution_pnl,top1_contribution_pct,top3_contribution_pct,top5_contribution_pct,worst1_contribution_pct,worst3_contribution_pct,worst5_contribution_pct,verdict,entity_label
0,m2k_only_baseline,strict_train_only,False,474.775,1.913,39,4,-133.125,-52.50,0.385,12.174,-11.625,0.440,25,0.000,0.000,0.474,1.058,1.502,-0.111,-0.261,-0.407,baseline,m2k_only_baseline
1,mgc_only_baseline,strict_train_only,False,1217.200,3.489,23,2,-265.000,-81.50,0.478,52.922,-11.500,0.625,16,1.000,1217.200,0.405,0.808,1.073,-0.067,-0.184,-0.262,baseline,mgc_only_baseline
2,raw_m2k_mgc_equal_weight,strict_train_only,False,845.988,2.677,62,4,-107.812,-40.75,0.419,13.645,-5.781,0.531,32,1.000,608.600,0.291,0.587,0.813,-0.048,-0.132,-0.192,baseline,raw_m2k_mgc_equal_weight
3,strict_best_regime_gated,strict_train_only,False,800.290,2.201,53,4,-167.625,-52.50,0.415,15.100,-10.636,0.452,31,0.523,438.015,0.308,0.654,0.938,-0.066,-0.157,-0.245,weak_watchlist,strict_best_regime_gated


### Selected regime rule by fold

,fold_id,rule_id,family,allocation_scheme,train_score,train_net_pnl,train_profit_factor,train_trades,mgc_retention_rate_train
0,fold_1,always_on__conditional_equal_weight,always_on,conditional_equal_weight,0.872,138.500,1.845,14,1.000
1,fold_2,always_on__conditional_equal_weight,always_on,conditional_equal_weight,0.184,112.625,1.148,41,1.000
2,fold_3,atr_pct_mid_q30_q70__conditional_inverse_vol,atr_pct_between,conditional_inverse_vol,1.780,543.283,1.635,57,0.400
3,fold_4,atr_pct_mid_q20_q80__conditional_equal_weight,atr_pct_between,conditional_equal_weight,2.094,706.400,1.651,81,0.629
4,fold_5,atr_pct_mid_q20_q80__conditional_equal_weight,atr_pct_between,conditional_equal_weight,3.500,974.900,2.236,72,0.630


In [9]:
display(Markdown("## 6. Portfolio Integration Readout"))

integration_view = portfolio_summary.loc[
    (portfolio_summary["scope"].astype(str) == "defined_oos")
    & (
        portfolio_summary["portfolio_name"].astype(str).isin(
            [
                BASELINE_PORTFOLIO,
                PULLBACK_PORTFOLIO,
                INTEGRATED_PORTFOLIO,
                INTEGRATED_M2K_ONLY,
            ]
        )
    )
].copy()
display(
    integration_view[
        [
            "portfolio_name",
            "net_pnl",
            "daily_sharpe",
            "sortino",
            "max_drawdown",
            "max_daily_loss",
            "profit_factor",
            "day_win_rate",
            "monthly_hit_rate",
            "verdict",
        ]
    ].round(3)
)

display(Markdown("### Incremental impact vs baseline"))
display(incremental_metrics.round(3))

display(Markdown("### Bootstrap robustness"))
display(
    bootstrap_summary[
        [
            "portfolio_name",
            "median_net_pnl",
            "p05_net_pnl",
            "p95_net_pnl",
            "probability_positive",
            "probability_drawdown_breach_2k",
            "probability_prop_pass",
        ]
    ].round(3)
)

display(Markdown("### Prop-firm constraint summary"))
display(prop_constraint_summary.round(3))


## 6. Portfolio Integration Readout

,portfolio_name,net_pnl,daily_sharpe,sortino,max_drawdown,max_daily_loss,profit_factor,day_win_rate,monthly_hit_rate,verdict
2,baseline_only,24714.000,2.088,2.587,-5950.500,-744.00,1.401,0.504,0.68,diversifier_watchlist
5,pullback_m2k_mgc_only,213.325,0.888,0.591,-64.875,-34.25,2.053,0.027,0.32,reject
11,baseline_plus_pullback_equal_notional,24927.325,2.108,2.615,-5877.875,-744.00,1.406,0.504,0.68,diversifier_watchlist
23,baseline_plus_m2k_only_equal_notional,24665.150,2.084,2.579,-5984.250,-744.00,1.401,0.504,0.68,reject


### Incremental impact vs baseline

,portfolio_name,incremental_net_pnl_vs_baseline,incremental_sharpe_vs_baseline,incremental_sortino_vs_baseline,incremental_max_drawdown_vs_baseline,incremental_monthly_hit_rate_vs_baseline
0,baseline_only,0.000,0.000,0.000,0.000,0.00
1,pullback_m2k_mgc_only,-24500.675,-1.200,-1.996,5885.625,-0.36
2,pullback_m2k_only,-24762.850,-2.371,-2.673,5795.250,-0.52
3,baseline_plus_pullback_equal_notional,213.325,0.020,0.028,72.625,0.00
4,baseline_plus_pullback_scaled_to_baseline_risk,247.252,0.020,0.028,64.625,0.00
5,baseline_plus_pullback_capped_50pct,106.663,0.010,0.014,36.312,0.00
6,baseline_plus_pullback_when_baseline_flat,0.000,0.000,0.000,0.000,0.00
7,baseline_plus_m2k_only_equal_notional,-48.850,-0.004,-0.007,-33.750,0.00


### Bootstrap robustness

,portfolio_name,median_net_pnl,p05_net_pnl,p95_net_pnl,probability_positive,probability_drawdown_breach_2k,probability_prop_pass
0,baseline_only,24739.750,8687.500,40725.075,0.996,0.998,0.554
1,pullback_m2k_mgc_only,204.606,-66.616,582.099,0.876,0.000,0.000
2,pullback_m2k_only,-79.850,-271.519,199.106,0.320,0.000,0.000
3,baseline_plus_pullback_equal_notional,24616.225,10071.524,40857.054,0.998,1.000,0.572
4,baseline_plus_pullback_scaled_to_baseline_risk,24677.266,7685.696,41027.188,0.998,1.000,0.614
5,baseline_plus_pullback_capped_50pct,24544.416,8678.979,40445.761,0.992,1.000,0.518
6,baseline_plus_pullback_when_baseline_flat,24670.000,9242.925,42037.575,0.996,1.000,0.556
7,baseline_plus_m2k_only_equal_notional,25083.425,8574.168,40587.575,0.994,1.000,0.552


### Prop-firm constraint summary

,portfolio_name,daily_loss_limit_breaches,max_daily_loss,max_drawdown,historical_prop_status,historical_days_to_pass,historical_days_to_fail,historical_final_profit_usd
0,baseline_only,0,-744.000,-2492.500,fail,NaN,17.0,-1984.000
1,pullback_m2k_mgc_only,0,-34.250,-39.875,expire,NaN,NaN,13.325
2,pullback_m2k_only,0,-36.000,-59.250,expire,NaN,NaN,47.150
3,baseline_plus_pullback_equal_notional,0,-744.000,-2492.500,fail,NaN,17.0,-1984.000
4,baseline_plus_pullback_scaled_to_baseline_risk,0,-745.013,-2495.892,fail,NaN,17.0,-1986.700
5,baseline_plus_pullback_capped_50pct,0,-744.000,-2492.500,fail,NaN,17.0,-1984.000
6,baseline_plus_pullback_when_baseline_flat,0,-744.000,-2492.500,fail,NaN,17.0,-1984.000
7,baseline_plus_m2k_only_equal_notional,0,-744.000,-2492.500,fail,NaN,17.0,-1984.000


In [10]:
display(Markdown("## 7. Trade Concentration and Seasonality"))

display(Markdown("### Trade concentration"))
display(survivor_trade_concentration.round(3))

monthly_focus = survivor_monthly_pnl.loc[
    survivor_monthly_pnl["entity_id"].astype(str).isin(PRIMARY_SYMBOLS)
].copy()
monthly_focus["month"] = pd.to_datetime(monthly_focus["month"], errors="coerce")

monthly_fig = px.bar(
    monthly_focus,
    x="month",
    y="pnl",
    color="entity_id",
    barmode="group",
    title="Monthly pnl by signal",
)
monthly_fig.update_layout(template=PLOT_TEMPLATE, width=1350, height=450)
monthly_fig.show()

if "year" in survivor_yearly_pnl.columns:
    yearly_focus = survivor_yearly_pnl.loc[survivor_yearly_pnl["entity_id"].astype(str).isin(PRIMARY_SYMBOLS)].copy()
    yearly_fig = px.bar(
        yearly_focus,
        x="year",
        y="pnl",
        color="entity_id",
        barmode="group",
        title="Yearly pnl by signal",
    )
    yearly_fig.update_layout(template=PLOT_TEMPLATE, width=1000, height=420)
    yearly_fig.show()


## 7. Trade Concentration and Seasonality

### Trade concentration

,entity_id,trade_count,total_pnl,top1_trade_pnl,top3_trade_pnl,top5_trade_pnl,top1_contribution_pct,top3_contribution_pct,top5_contribution_pct,worst1_trade_pnl,worst3_trade_pnl,worst5_trade_pnl,worst1_contribution_pct,worst3_contribution_pct,worst5_contribution_pct
0,M2K,39,474.775,225.00,502.25,713.025,0.474,1.058,1.502,-52.5,-124.125,-193.125,-0.111,-0.261,-0.407
1,MGC,23,1217.200,492.50,984.00,1306.000,0.405,0.808,1.073,-81.5,-223.500,-319.500,-0.067,-0.184,-0.262
2,MNQ,116,-878.125,667.65,1527.15,2070.250,-0.760,-1.739,-2.358,-269.5,-728.500,-1131.000,0.307,0.830,1.288


In [11]:
display(Markdown("## 8. Conclusion"))

lines = [
    f"- `M2K 1H` reste le signal coeur a conserver en watchlist : net `{fmt_money(m2k_row['net_pnl'])}` | PF `{fmt_float(m2k_row['profit_factor'])}` | verdict `{m2k_row['verdict']}`.",
    f"- `MGC 1H` n'est pas promu seul : net `{fmt_money(mgc_row['net_pnl'])}` mais seulement `{int(mgc_row['positive_folds'])}/{int(mgc_row['fold_count'])}` folds positifs, donc verdict `{mgc_row['verdict']}`.",
    f"- Le meilleur sleeve strict a garder comme reference est `M2K + MGC` equal-weight : net `{fmt_money(strict_sleeve_row['net_pnl'])}` | PF `{fmt_float(strict_sleeve_row['profit_factor'])}` | verdict `{strict_sleeve_row['verdict']}`.",
    f"- Le gating regime n'a pas amene de gain robuste : `{fmt_money(best_regime_row['net_pnl'])}` versus raw strict `{fmt_money(strict_sleeve_row['net_pnl'])}`.",
    f"- Comme overlay portefeuille, le sleeve ajoute `{fmt_money(incremental_equal['incremental_net_pnl_vs_baseline'])}` au baseline avec une correlation faible `{fmt_float(correlation_equal['correlation'])}`, donc `diversifier_watchlist` plutot que candidat deployable.",
    "- Si tu veux la suite naturelle, on peut maintenant te construire le replay scene-by-scene des trades selectionnes avec bougies 1min, niveaux stop/target, ordres et etat regime.",
]
display(Markdown("\n".join(lines)))


## 8. Conclusion

- `M2K 1H` reste le signal coeur a conserver en watchlist : net `474.8 USD` | PF `1.913` | verdict `weak_watchlist`.
- `MGC 1H` n'est pas promu seul : net `1,217.2 USD` mais seulement `2/4` folds positifs, donc verdict `reject`.
- Le meilleur sleeve strict a garder comme reference est `M2K + MGC` equal-weight : net `846.0 USD` | PF `2.677` | verdict `watchlist`.
- Le gating regime n'a pas amene de gain robuste : `800.3 USD` versus raw strict `846.0 USD`.
- Comme overlay portefeuille, le sleeve ajoute `213.3 USD` au baseline avec une correlation faible `-0.056`, donc `diversifier_watchlist` plutot que candidat deployable.
- Si tu veux la suite naturelle, on peut maintenant te construire le replay scene-by-scene des trades selectionnes avec bougies 1min, niveaux stop/target, ordres et etat regime.